In [42]:
import xgboost as xgb
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

In [43]:
df_musteri = pd.DataFrame({
    'Musteri_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Yasi': ['25', '45', '35', '50', '23', '40', '60', '20', '30', '55'],  
    'Sehir': ['Istanbul', 'Ankara', 'Istanbul', 'Izmir', 'Ankara', 'Istanbul', 'Izmir', 'Ankara', 'Istanbul', 'Izmir']
})


df_finans = pd.DataFrame({
    'Musteri_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Gelir': [25000.0, np.nan, 45000.0, 18000.0, np.nan, 55000.0, 12000.0, 22000.0, 60000.0, 15000.0],
    'Kredi_Skoru': [550, 720, 680, 500, 580, 790, 480, 610, 810, 520],
    'Borclu_Mu': [1, 0, 0, 1, 1, 0, 1, 0, 0, 1]  
})

In [64]:
df=pd.merge(df_musteri,df_finans,on="Musteri_ID")
df.drop("Musteri_ID",axis=1)
df.head(3)

,Musteri_ID,Yasi,Sehir,Gelir,Kredi_Skoru,Borclu_Mu
0,101,25,Istanbul,25000.0,550,1
1,102,45,Ankara,NaN,720,0
2,103,35,Istanbul,45000.0,680,0


In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Musteri_ID   10 non-null     int64  
 1   Yasi         10 non-null     str    
 2   Sehir        10 non-null     str    
 3   Gelir        8 non-null      float64
 4   Kredi_Skoru  10 non-null     int64  
 5   Borclu_Mu    10 non-null     int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 612.0 bytes


In [66]:
df["Yasi"]=df["Yasi"].astype(int)

In [67]:
gelir_ortalama=df.groupby("Sehir")["Gelir"].transform("mean")

In [68]:
df["Gelir"]=df["Gelir"].fillna(gelir_ortalama)

In [69]:
df["Yuksek_Risk_Sinyali"]=np.where((df["Gelir"]<25000) & (df["Kredi_Skoru"]<600),1,0)
df=pd.get_dummies(df,columns=["Sehir"],drop_first=True)

In [70]:
df.head(3)

,Musteri_ID,Yasi,Gelir,Kredi_Skoru,Borclu_Mu,Yuksek_Risk_Sinyali,Sehir_Istanbul,Sehir_Izmir
0,101,25,25000.0,550,1,0,True,False
1,102,45,22000.0,720,0,0,False,False
2,103,35,45000.0,680,0,0,True,False


In [71]:
y=df["Borclu_Mu"]
x=df.drop("Borclu_Mu",axis=1)

In [72]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,train_size=0.8)
rf=RandomForestClassifier()
model=rf.fit(x_train,y_train)
model.score(x_test,y_test)


1.0

In [73]:
def kredi_modeli_egit(df:pd.DataFrame)->float:
    return model.score(x_test,y_test)

In [74]:
model_xgb=xgb.XGBClassifier(n_estimators=100,learning_rate=0.1,random_state=42)

In [75]:
model2=model_xgb.fit(x_train,y_train)
model2.score(x_test,y_test)

0.0

In [76]:
y_pred=model2.predict(x_test)

In [77]:
confusion_matrix(y_test,y_pred)

array([[0, 2],
       [0, 0]])

In [63]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       2.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



c:\Users\Merve\Desktop\PythonProjects\python-for-ai\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Merve\Desktop\PythonProjects\python-for-ai\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Merve\Desktop\PythonProjects\python-for-ai\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av